In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
experiment_dirs = {
    "Baseline No FINO": "/data/raymond.biju/nanopath/20260826-pf_baseline",
    "XAttn JEPA": "/data/raymond.biju/nanopath/20260824-xaj_lmbd0_ctx12_8_4",
    "Factorized f3p768": "/data/raymond.biju/nanopath/20260827-pf_f3_p768_bsN",
    "XAttn + Factorized Head": "/data/raymond.biju/nanopath/20260828-xajpf_f3_bs384",
}
COLORS = dict(zip(experiment_dirs, ["#4c78a8", "#f58518", "#54a24b", "#e45756"]))


def load_run(path):
    rows = [json.loads(line) for line in Path(path, "metrics.jsonl").read_text().splitlines() if line.strip()]
    train = pd.DataFrame([r for r in rows if "dino" in r])
    val = pd.DataFrame([r for r in rows if any(k.startswith("val_") for k in r)])
    summary = json.loads(Path(path, "summary.json").read_text())
    return train, val, summary


runs = {name: load_run(path) for name, path in experiment_dirs.items()}

setup = pd.DataFrame([
    {
        "run": name,
        "batch_size": s.get("batch_size"),
        "lr": s.get("lr"),
        "head_factors": s.get("head_factors"),
        "head_prototypes": s.get("head_prototypes"),
        "log_points": len(tr),
        "final_sample_frac": round(float(tr["sample_fraction"].iloc[-1]), 3),
    }
    for name, (tr, va, s) in runs.items()
])
setup

In [ ]:
def smooth(y, w=9):
    y = np.asarray(y, dtype=float)
    if len(y) < w:
        return y
    return np.convolve(y, np.ones(w) / w, mode="same")


def panel(ax, col, title, logy=False, source="train", smooth_w=9):
    for name, (tr, va, _) in runs.items():
        df = tr if source == "train" else va
        if col not in df or df.empty:
            continue
        x = df["sample_fraction"] if source == "train" else df["step"] / df["step"].max()
        ax.plot(x, smooth(df[col], smooth_w), label=name, color=COLORS[name], lw=1.4)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("sample fraction")
    if logy:
        ax.set_yscale("log")
    ax.grid(alpha=0.25)


fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
panel(axes[0, 0], "dino", "DINO")
panel(axes[0, 1], "jepa", "JEPA")
panel(axes[0, 2], "kde", "KDE")
panel(axes[1, 0], "total", "total")
panel(axes[1, 1], "grad_norm", "grad_norm (clip=3.0)")
axes[1, 1].axhline(3.0, color="k", ls="--", lw=1, alpha=0.6)
panel(axes[1, 2], "lr", "lr")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Tier 0 — raw loss terms, x-axis = sample fraction (steps are not comparable: batch sizes differ)", fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for name, (tr, _, _) in runs.items():
    axes[0].plot(tr["sample_fraction"], smooth(tr["dino"] / tr["jepa"]), label=name, color=COLORS[name], lw=1.4)
    axes[1].plot(tr["sample_fraction"], smooth(tr["dino"] / tr["dino"].iloc[:5].mean()), color=COLORS[name], lw=1.4)
    axes[2].plot(tr["sample_fraction"], smooth(tr["jepa"] / tr["jepa"].iloc[:5].mean()), color=COLORS[name], lw=1.4)

for ax, t in zip(axes, ["dino / jepa  (relative pull on the trunk)", "dino, normalised to its own start", "jepa, normalised to its own start"]):
    ax.set_title(t, fontsize=10)
    ax.set_xlabel("sample fraction")
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

# DINO's magnitude scales with the code space: F*log(K_per_factor) vs log(K) unfactorised.
print("expected DINO scale at uniform assignment")
for name, (_, _, s) in runs.items():
    f, p = s.get("head_factors") or 1, s.get("head_prototypes") or 131072
    print(f"  {name:26s} F={f}  K/factor={p // f:6d}  F*log(K) = {f * np.log(p / f):5.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, term in enumerate(["dino", "jepa", "kde"]):
    for name, (tr, va, _) in runs.items():
        if va.empty:
            continue
        frac = va["step"] / va["step"].max()
        axes[i].plot(frac, va[f"val_{term}"], color=COLORS[name], lw=1.5, label=f"{name} (val)")
        axes[i].plot(tr["sample_fraction"], smooth(tr[term]), color=COLORS[name], lw=1.0, ls=":", alpha=0.7)
    axes[i].set_title(f"{term}: val (solid) vs train (dotted)", fontsize=10)
    axes[i].set_xlabel("fraction of run")
    axes[i].grid(alpha=0.25)
axes[0].legend(fontsize=7)
fig.tight_layout()
plt.show()

tail = pd.DataFrame([
    {
        "run": name,
        **{f"train_{c}": round(float(tr[c].tail(10).mean()), 4) for c in ("dino", "jepa", "kde", "total", "grad_norm")},
        **{f"val_{c}": (round(float(va[f"val_{c}"].tail(3).mean()), 4) if not va.empty else None) for c in ("dino", "jepa", "kde")},
        "clip_frac": round(float((tr["grad_norm"] > 3.0).mean()), 3),
        "dino/jepa": round(float((tr["dino"] / tr["jepa"]).tail(10).mean()), 3),
    }
    for name, (tr, va, _) in runs.items()
])
tail

In [ ]:
# Probe scores: the reported number, plus its components, so a single weak task is visible.
probe = {}
for name, path in experiment_dirs.items():
    rows = [json.loads(l) for l in open(f"{path}/metrics.jsonl") if l.strip()]
    hit = [r for r in rows if "mean_probe_score" in r]
    probe[name] = hit[-1] if hit else {}

keys = ["mean_probe_score", "linear_mean_f1", "knn_mean_f1", "fewshot_mean_f1", "auc_mean"]
per_task = sorted({k for r in probe.values() for k in r if k.startswith("probe_") and k.endswith("_score")})
scores = pd.DataFrame([{"run": n, **{k: r.get(k) for k in keys + per_task}} for n, r in probe.items()]).set_index("run")

display(scores[keys].round(4))

d = scores[per_task].astype(float)
delta = (d - d.loc["Baseline No FINO"]).drop(index="Baseline No FINO")
fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(per_task)), 4))
delta.T.plot.bar(ax=ax, color=[COLORS[i] for i in delta.index], width=0.8)
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("score - baseline")
ax.set_title("Per-task delta vs baseline: is the combined run uniformly worse, or is one task dragging it?", fontsize=10)
ax.set_xticklabels([t.replace("probe_", "").replace("_score", "") for t in delta.columns], rotation=45, ha="right", fontsize=8)
ax.grid(alpha=0.25, axis="y")
fig.tight_layout()
plt.show()

print("spread across the four runs, per task (a task whose spread ~ the 0.013 headline gap is where the signal is):")
print(d.max() - d.min())